# Batch Scoring — Fraud Detection

Scores every transaction with the trained model to produce fraud-risk
predictions and a per-merchant-category risk summary.

**Reads:** `gold_ml_features` + saved model  **Writes:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count, isnan, udf,
    sum as spark_sum, round as spark_round
)
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassificationModel

spark = SparkSession.builder.getOrCreate()
model = RandomForestClassificationModel.load('Files/models/fraud_detection_rf')
df = spark.read.table('gold_ml_features')
print(f'Scoring {df.count():,} feature rows')

In [ ]:
for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

numeric_features = [
    'amount', 'log_amount', 'transaction_hour', 'is_night', 'is_international',
    'is_high_value', 'balance', 'credit_limit', 'credit_utilisation_pct',
]
cat_cols = ['transaction_type', 'merchant_category', 'channel', 'country',
            'account_type', 'age_group', 'segment', 'region', 'risk_tier']
indexed_df = df
for c in cat_cols:
    idx_col = f'{c}_idx'
    indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid='keep')
    indexed_df = indexer.fit(indexed_df).transform(indexed_df)

all_features = numeric_features + [f'{c}_idx' for c in cat_cols]
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(indexed_df)

In [ ]:
scored = model.transform(model_df)
extract_prob = udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.0, DoubleType())

predictions = (
    scored
    .withColumn('fraud_probability', spark_round(extract_prob(col('probability')), 4))
    .withColumn('predicted_fraud', col('prediction').cast('int'))
    .withColumn('risk_level',
        when(col('fraud_probability') > 0.8, 'critical')
        .when(col('fraud_probability') > 0.6, 'high')
        .when(col('fraud_probability') > 0.4, 'medium')
        .otherwise('low'))
    .withColumn('scored_at', current_timestamp())
    .select(
        'transaction_id', 'account_id', 'customer_id', 'transaction_date',
        'merchant_category', 'channel', 'country', 'segment', 'region',
        'amount', 'had_fraud', 'predicted_fraud', 'fraud_probability', 'risk_level',
        'scored_at')
)
predictions.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count():,} rows')
predictions.groupBy('risk_level').count().orderBy('count', ascending=False).show()

In [ ]:
# Per merchant-category fraud risk summary
summary = (
    predictions
    .groupBy('merchant_category')
    .agg(
        count('*').alias('total_transactions'),
        spark_sum('predicted_fraud').alias('predicted_fraud_count'),
        spark_sum('had_fraud').alias('actual_fraud_count'),
        spark_round(avg('fraud_probability'), 4).alias('avg_fraud_probability'),
        spark_round(spark_sum('amount'), 2).alias('total_amount'),
    )
    .withColumn('fraud_rate', spark_round(col('predicted_fraud_count') / col('total_transactions') * 100, 1))
    .withColumn('overall_risk',
        when(col('avg_fraud_probability') > 0.6, 'high')
        .when(col('avg_fraud_probability') > 0.3, 'medium')
        .otherwise('low'))
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('avg_fraud_probability').desc())
)
summary.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_summary')
print(f'Merchant-category risk summary: {summary.count()} rows')
summary.show(15, truncate=False)

In [ ]:
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('All Gold ML tables optimized')